In [ ]:
import re
import nltk
import pandas as pd
import unicodedata
import contractions
import requests 
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Anthony\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Anthony\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Anthony\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Anthony\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
stopword_list = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

def strip_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

def remove_accented_chars(text):
    text = unicodedata.normalize('NFKD', text)
    return text.encode('ascii', 'ignore').decode('utf-8')

def expand_contractions(text):
    return contractions.fix(text)

def remove_special_characters(text, remove_digits=True):
    pattern = r'[^a-zA-Z\s]' if remove_digits else r'[^a-zA-Z0-9\s]'
    return re.sub(pattern, '', text)

def remove_stopwords(text, is_lower_case=False):
    tokens = nltk.word_tokenize(text)
    tokens = [token for token in tokens if token not in stopword_list]
    return ' '.join(tokens)

def lemmatize_text(text):
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(tokens)

In [138]:
def normalize_corpus(corpus, html_stripping=True, contraction_expansion=True,
                     accented_char_removal=True, text_lower_case=True,
                     text_lemmatization=True, special_char_removal=True,
                     stopword_removal=True, remove_digits=True):
    normalized_corpus = []
    for doc in corpus:
        if html_stripping:
            doc = strip_html_tags(doc)
        if accented_char_removal:
            doc = remove_accented_chars(doc)
        if contraction_expansion:
            doc = expand_contractions(doc)       
        if text_lower_case:
            doc = doc.lower()
            doc = re.sub(r'[\r\n]+', ' ', doc)
        if text_lemmatization:
            doc = lemmatize_text(doc)
        if special_char_removal:
            special_char_pattern = re.compile(r'([{}\(\)\-!])')
            doc = special_char_pattern.sub(r' ', doc)
            doc = remove_special_characters(doc, remove_digits=remove_digits)
        doc = re.sub(r' +', ' ', doc).strip()
        if stopword_removal:
            doc = remove_stopwords(doc, is_lower_case=text_lower_case)
        normalized_corpus.append(doc)
    return normalized_corpus

In [139]:
documents = [
    #Social Media Post
    "It’s great to see India’s leadership in scientific innovation and vaccine manufacturing capability as the world works to end the COVID-19 pandemic",

    #Product Review
    "Bright-sounding tone & easy-to-drive signal from a dynamic microphone are the signature characteristics."
    "High-density flocked pop filter & metal shock mount do their job in functionality and aesthetics.",

    #News Snippet
    "Iran’s response: Foreign Minister Abbas Araghchi acknowledged that messages have been exchanged with the US through mediators"
    "but said that Washington’s shift in tone amounted to an acknowledgment of failure after previously demanding Tehran’s “unconditional surrender.",
]


version_a = normalize_corpus(documents,
    html_stripping=True,
    contraction_expansion=True,
    accented_char_removal=True,
    text_lower_case=True,
    text_lemmatization=True,       
    stopword_removal=True,          
    remove_digits=True)

version_b = normalize_corpus(documents,
    html_stripping=True,
    contraction_expansion=True,
    accented_char_removal=True,
    text_lower_case=True,
    text_lemmatization=False,     
    special_char_removal=True,
    stopword_removal=False,        
    remove_digits=True)

labels = [
    "Social Media Post",
    "Product Review",
    "News Snippet"
]

for i, label in enumerate(labels):

    print("\n")
    print(f"  {label}")
    print("\n")
    print(f"  ORIGINAL  : {documents[i][:100]}")
    print(f"  VERSION A : {version_a[i]}")
    print(f"  VERSION B : {version_b[i]}")
   



  Social Media Post


  ORIGINAL  : It’s great to see India’s leadership in scientific innovation and vaccine manufacturing capability a
  VERSION A : great see india leadership scientific innovation vaccine manufacturing capability world work end covid pandemic
  VERSION B : its great to see indias leadership in scientific innovation and vaccine manufacturing capability as the world works to end the covid pandemic


  Product Review


  ORIGINAL  : Bright-sounding tone & easy-to-drive signal from a dynamic microphone are the signature characterist
  VERSION A : bright sounding tone easy drive signal dynamic microphone signature characteristicshigh density flocked pop filter metal shock mount job functionality aesthetic
  VERSION B : bright sounding tone easy to drive signal from a dynamic microphone are the signature characteristicshigh density flocked pop filter metal shock mount do their job in functionality and aesthetics


  News Snippet


  ORIGINAL  : Iran’s response: Foreign 